Imports

In [1]:
# imports
from esdl import esdl
from esdl.esdl_handler import EnergySystemHandler
import pandas as pd
import numpy as np
import numpy_financial as npf
from decimal import Decimal, ROUND_HALF_UP
import matplotlib.pyplot as plt

In [2]:
# Import the business case runner
from set_up.generic_business_case_runner import *

In [3]:
# This imports all input variables from EL_input_data.py
from assets.OnEL_input_data import *

# import all data from ESDL
from esdl_files.NSE_get_data_from_ESDL import *



Active Scenario: most_likely (Index: 1)
All variables saved to NSE_get_data_from_ESDL.pkl


In [4]:
asset_parameters

name,power,efficiency,investment_costs,fixed_opex,variable_opex,wacc
TNVDW,700000000.0,,1750.0,2.25,5.0,8.5
Electrolyzer,500.0,0.6,2000.0,2.0,0.0,8.25
Offtaker,24900000.0,,0.0,0.0,0.0,10.5


Set up calendar_year, business_case_year, operations_years, and decommissioning_years lists

In [5]:
from set_up.construct_timelines import (
    construct_calendar_year_list,
    construct_business_case_year_list,
    construct_operations_years_list,
    construct_decommissioning_years_list
)

Start business case analysis

Construction phase

In [6]:
def construction_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the construction phase'''

    # 1. Get the parameters we need
    capex = kwargs['capex']
    duration_construction = kwargs['duration_construction']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)

    # 3. Construct df
    row_names = ['capex', 'total_cashflow_investment']
    df_construction_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_construction_phase.columns.name = "construction_phase"

    # 4. Calculate yearly CAPEX value 
    yearly_capex = -capex / duration_construction

    # 5. Add CAPEX numbers to construction years
    df_construction_phase.loc['capex'] = np.where(
        df_construction_phase.columns.isin(construction_years_list),
        yearly_capex, 0.0)

    # 6. Calculate total investments
    # at this moment only one cashflow is in construction phase, can be expanded later
    total_cashflow_investment = df_construction_phase.loc['capex']
    df_construction_phase.loc['total_cashflow_investment'] = total_cashflow_investment
    
    return df_construction_phase

In [7]:
df_construction_phase = construction_phase(**onel_parameters)
df_construction_phase.style.format(precision=2)

construction_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
capex,0.00,-421.92,-421.92,-421.92,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
total_cashflow_investment,0.00,-421.92,-421.92,-421.92,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


Operational phase

In [8]:
def operational_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the operational phase'''

    # 1. Get the parameters we need
    inflation = kwargs['inflation']
    opex_df = kwargs['opex_df']
    annual_electricity_costs_ppa = kwargs['annual_electricity_costs_ppa']
    annual_electricity_costs_grid = kwargs['annual_electricity_costs_grid']
    electricity_grid_connection = kwargs['electricity_grid_connection']
    h2_storage_costs = kwargs['h2_storage_costs']
    entry_tariff_h2_network = kwargs['entry_tariff_h2_network']
    electricity_tax = kwargs['electricity_tax']
    df_stack_replacement_costs = kwargs['df_stack_replacement_costs']
    hydrogen_revenues = kwargs['hydrogen_revenues']
    hwi_revenues = kwargs['hwi_revenues']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    operational_years = construct_operations_years_list(**kwargs)
    last_construction_year = construction_years_list[-1]
    business_case_years = construct_business_case_year_list(**kwargs)

    # 3. construct df
    row_names = [
        'opex', 'purchase_electricity_ppa', 'purchase_electricity_grid', 
        'electricity_grid_connection', 'h2_storage_costs', 'entry_tariff_h2_network', 
        'electricity_tax','stack_replacement', 
        'total_outflow_opex', 'h2_revenues', 'hwi_revenues', 
        'total_revenues_opex', 'net_cashflow_operations'
    ]

    df_operational_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_operational_phase.columns.name = "operational_phase"

    # 4. Calculate inflation factors array 
    inf_factors = (1 + inflation) ** business_case_years
    # Turn it into a series to give the calendar years as index
    # This way we can use it in our loop over the operational years
    inf_factor_series = pd.Series(inf_factors, index=calendar_years)


    # 5. Calculate the outflows
    for x in operational_years:

        # Get the inflation factor for that year
        inf_factor = inf_factor_series[x]

        # Outflows
        
        # opex, annual electricity costs ppa/grid are dfs
        df_operational_phase.loc['opex', x] = -opex_df.loc['opex',x] * inf_factor
        df_operational_phase.loc['purchase_electricity_ppa', x] = (
            -annual_electricity_costs_ppa.loc['onel_annual_electricity_costs_ppa',x] * 
            inf_factor)
        df_operational_phase.loc['purchase_electricity_grid', x] = (
            -annual_electricity_costs_grid.loc['onel_annual_electricity_costs_grid',x] * 
            inf_factor)
        # electricity grid connection is a value    
        df_operational_phase.loc['electricity_grid_connection', x] = (
            -electricity_grid_connection * inf_factor)
        # h2 storage costs is a df
        df_operational_phase.loc['h2_storage_costs', x] = (
            -h2_storage_costs.loc['onel_h2_storage_costs',x] * inf_factor)
        # entry tariff h2 network is a df
        df_operational_phase.loc['entry_tariff_h2_network',x] = (
            -entry_tariff_h2_network.loc['onel_entry_tariff_h2_network',x] * inf_factor)
        # electricity tax is a df
        df_operational_phase.loc['electricity_tax',x] = (
            -electricity_tax.loc['onel_electricity_tax',x] * inf_factor)


        # 6. Calculate stack replacement, every 5 years after construction end
        if (x - last_construction_year) % 5 == 0:    
            df_operational_phase.loc['stack_replacement',x] = (
                -df_stack_replacement_costs.loc['stack_replacement', x] * inf_factor)


        # 7. Calculate revenues
        df_operational_phase.loc['h2_revenues', x] = (
            hydrogen_revenues.loc['onel_h2_revenues', x] * inf_factor)
        df_operational_phase.loc['hwi_revenues', x] = (
            hwi_revenues.loc['onel_hwi_revenues', x] * inf_factor)

            
    # 8. Calculate totals
    outflow_rows = [
        'opex', 'purchase_electricity_ppa', 'purchase_electricity_grid', 
        'electricity_grid_connection', 'h2_storage_costs', 'entry_tariff_h2_network', 
        'electricity_tax','stack_replacement',
    ]
    df_operational_phase.loc['total_outflow_opex'] = df_operational_phase.loc[outflow_rows].sum()

    revenue_rows = ['h2_revenues', 'hwi_revenues']
    df_operational_phase.loc['total_revenues_opex'] = df_operational_phase.loc[revenue_rows].sum()

    # Net cashflow from operation (=EBITDA)
    df_operational_phase.loc['net_cashflow_operations'] = (
        df_operational_phase.loc['total_outflow_opex'] + 
        df_operational_phase.loc['total_revenues_opex'])

    
    return df_operational_phase      

In [9]:
df_operational_phase = operational_phase(**onel_parameters)
df_operational_phase.style.format(precision=2)

operational_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
opex,0.00,0.00,0.00,0.00,-39.96,-39.93,-39.88,-39.81,-39.72,-39.62,-39.49,-39.34,-39.18,-38.98,-39.76,-40.56,-41.37,-42.20,-43.04,-43.90,-44.78,-45.68,-46.59,-47.52,-48.47,-49.44,-50.43,-51.44,-52.47,0.00,0.00
purchase_electricity_ppa,0.00,0.00,0.00,0.00,-103.42,-105.44,-107.49,-109.59,-111.73,-113.91,-116.13,-118.39,-120.70,-123.05,-117.26,-111.18,-104.81,-98.14,-91.17,-83.87,-76.25,-68.29,-59.98,-51.31,-52.33,-53.38,-54.45,-55.54,-56.65,0.00,0.00
purchase_electricity_grid,0.00,0.00,0.00,0.00,-10.20,-12.18,-14.24,-16.38,-18.59,-20.89,-23.27,-25.74,-28.30,-30.96,-28.56,-26.06,-23.44,-20.71,-17.86,-14.89,-11.80,-8.57,-5.21,-1.71,-1.75,-1.78,-1.82,-1.86,-1.89,0.00,0.00
electricity_grid_connection,0.00,0.00,0.00,0.00,-78.10,-79.66,-81.25,-82.88,-84.54,-86.23,-87.95,-89.71,-91.50,-93.33,-95.20,-97.10,-99.05,-101.03,-103.05,-105.11,-107.21,-109.36,-111.54,-113.77,-116.05,-118.37,-120.74,-123.15,-125.61,0.00,0.00
h2_storage_costs,0.00,0.00,0.00,0.00,-2.03,-2.07,-2.11,-2.15,-2.20,-2.24,-2.28,-2.33,-2.38,-2.42,-2.53,-2.65,-2.77,-2.89,-3.01,-3.14,-3.27,-3.41,-3.55,-3.69,-3.77,-3.84,-3.92,-4.00,-4.08,0.00,0.00
entry_tariff_h2_network,0.00,0.00,0.00,0.00,-5.72,-5.83,-5.95,-6.07,-6.19,-6.31,-6.44,-6.57,-6.70,-6.83,-6.97,-7.11,-7.25,-7.40,-7.54,-7.70,-7.85,-8.01,-8.17,-8.33,-8.50,-8.67,-8.84,-9.02,-9.20,0.00,0.00
electricity_tax,0.00,0.00,0.00,0.00,-6.20,-6.33,-6.45,-6.58,-6.72,-6.85,-6.99,-7.13,-7.27,-7.41,-7.56,-7.71,-7.87,-8.03,-8.19,-8.35,-8.52,-8.69,-8.86,-9.04,-9.22,-9.40,-9.59,-9.78,-9.98,0.00,0.00
stack_replacement,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-200.21,0.00,0.00,0.00,0.00,-196.49,0.00,0.00,0.00,0.00,-216.94,0.00,0.00,0.00,0.00,-239.52,0.00,0.00,0.00,0.00,-264.44,0.00,0.00
total_outflow_opex,0.00,0.00,0.00,0.00,-245.63,-251.44,-257.38,-263.46,-469.89,-276.04,-282.55,-289.21,-296.03,-499.48,-297.85,-292.37,-286.55,-280.39,-490.80,-266.96,-259.68,-251.99,-243.90,-474.89,-240.09,-244.89,-249.78,-254.78,-524.32,0.00,0.00
h2_revenues,0.00,0.00,0.00,0.00,126.46,129.66,132.93,136.28,139.71,143.23,146.83,150.51,154.29,158.15,151.74,145.01,137.94,130.54,122.78,114.66,106.17,97.30,88.02,78.34,79.90,81.50,83.13,84.79,86.49,0.00,0.00


Decommissioning phase

In [10]:
def decommissioning_phase(**kwargs):
    ''' This function creates a dataframe that contains all cashflows in the decommissioning phase'''
    
    # 1. Get the parameters we need
    capex = kwargs['capex']
    decommissioning_percentage = kwargs['decommissioning_percentage']
    inflation = kwargs['inflation']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    decommissioning_years = construct_decommissioning_years_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)

    # 3. construct df
    row_names = ['decommissioning_costs', 'total_cashflow_decommissioning']
    df_decommissioning_phase = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_decommissioning_phase.columns.name = "decommissioning_phase"

    # 4. Calculate inflation factors
    inf_factors = (1 + inflation) ** business_case_years
    # Turn it into a series to give the calendar years as index
    # This way we can use it in our loop over the operational years
    inf_factor_series = pd.Series(inf_factors, index=calendar_years)

    # 5. Calculate decommissioning costs
    total_decommissioning_costs = capex * decommissioning_percentage
    yearly_decommissioning_costs = total_decommissioning_costs / len(decommissioning_years)

    df_decommissioning_phase.loc['decommissioning_costs'] = np.where(
        df_decommissioning_phase.columns.isin(decommissioning_years),
        -yearly_decommissioning_costs * inf_factor_series,
        0.0
    )

    # 4. Calculate totals
    df_decommissioning_phase.loc['total_cashflow_decommissioning'] = df_decommissioning_phase.loc['decommissioning_costs'] 
    
    
    
    return df_decommissioning_phase

In [11]:
df_decommissioning_phase = decommissioning_phase(**onel_parameters)
df_decommissioning_phase.style.format(precision=2)

decommissioning_phase,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
decommissioning_costs,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-22.48,-22.93
total_cashflow_decommissioning,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-22.48,-22.93


Taxes & profits - part 1

In [12]:
from finances.taxes_debt_loans import taxes_and_profits_part1

df_taxes_and_profits_part1 = taxes_and_profits_part1(df_operational_phase,**onel_parameters)
df_taxes_and_profits_part1.style.format(precision=2)

taxes_and_profits,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
depreciation,0.00,0.00,0.00,0.00,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,0.00,0.00
ebit,0.00,0.00,0.00,0.00,131.22,134.62,138.09,141.63,-54.98,148.90,152.63,156.44,160.32,-32.22,170.20,176.28,182.52,188.91,-21.46,202.20,209.10,216.17,223.42,-8.67,236.48,242.22,248.08,254.05,-4.30,0.00,0.00


Debt and loan - part 1

In [13]:
from finances.taxes_debt_loans import debt_and_loan_part1

df_debt_and_loan_part1 = debt_and_loan_part1(**onel_parameters)
df_debt_and_loan_part1.style.format(precision=2)

debt_and_loan,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
begin_of_year,0.00,0.00,210.96,421.92,632.88,603.55,572.75,540.42,506.46,470.81,433.38,394.08,352.81,309.48,263.98,216.21,166.04,113.37,58.07,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
drawdown,0.00,210.96,210.96,210.96,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_capital,0.00,0.00,0.00,0.00,-29.33,-30.80,-32.34,-33.95,-35.65,-37.43,-39.30,-41.27,-43.33,-45.50,-47.77,-50.16,-52.67,-55.30,-58.07,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
end_of_year,0.00,210.96,421.92,632.88,603.55,572.75,540.42,506.46,470.81,433.38,394.08,352.81,309.48,263.98,216.21,166.04,113.37,58.07,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_interest,0.00,0.00,0.00,0.00,-31.64,-30.18,-28.64,-27.02,-25.32,-23.54,-21.67,-19.70,-17.64,-15.47,-13.20,-10.81,-8.30,-5.67,-2.90,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00


Taxes & profits - part 2

In [14]:
from finances.taxes_debt_loans import taxes_and_profits_part2

df_taxes_and_profits_part2 = taxes_and_profits_part2(df_operational_phase,
                                                     df_taxes_and_profits_part1,
                                                     df_debt_and_loan_part1,
                                                     **onel_parameters)
df_taxes_and_profits_part2.style.format(precision=2)

taxes_and_profits,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
depreciation,0.00,0.00,0.00,0.00,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,-50.63,0.00,0.00
ebit,0.00,0.00,0.00,0.00,131.22,134.62,138.09,141.63,-54.98,148.90,152.63,156.44,160.32,-32.22,170.20,176.28,182.52,188.91,-21.46,202.20,209.10,216.17,223.42,-8.67,236.48,242.22,248.08,254.05,-4.30,0.00,0.00
interest_costs,0.00,0.00,0.00,0.00,-31.64,-30.18,-28.64,-27.02,-25.32,-23.54,-21.67,-19.70,-17.64,-15.47,-13.20,-10.81,-8.30,-5.67,-2.90,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
ebt,0.00,0.00,0.00,0.00,99.57,104.45,109.46,114.61,-80.30,125.36,130.96,136.74,142.68,-47.70,157.00,165.47,174.21,183.25,-24.37,202.20,209.10,216.17,223.42,-8.67,236.48,242.22,248.08,254.05,-4.30,0.00,0.00
tax_expenses,0.00,0.00,0.00,0.00,-25.69,-26.95,-28.24,-29.57,0.00,-32.34,-33.79,-35.28,-36.81,0.00,-40.51,-42.69,-44.95,-47.28,0.00,-52.17,-53.95,-55.77,-57.64,0.00,-61.01,-62.49,-64.00,-65.55,0.00,0.00,0.00
net_profits,0.00,0.00,0.00,0.00,73.88,77.50,81.22,85.04,-80.30,93.01,97.18,101.46,105.87,-47.70,116.49,122.78,129.27,135.97,-24.37,150.03,155.15,160.40,165.77,-8.67,175.47,179.73,184.07,188.51,-4.30,0.00,0.00


Debt & loan - part 2

In [15]:
from finances.taxes_debt_loans import debt_and_loan_part2

df_debt_and_loan_part2 = debt_and_loan_part2(df_operational_phase,
                                             df_debt_and_loan_part1,
                                             df_taxes_and_profits_part2,
                                             **onel_parameters)

df_debt_and_loan_part2.style.format(precision=2)

debt_and_loan,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
begin_of_year,0.00,0.00,210.96,421.92,632.88,603.55,572.75,540.42,506.46,470.81,433.38,394.08,352.81,309.48,263.98,216.21,166.04,113.37,58.07,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
drawdown,0.00,210.96,210.96,210.96,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_capital,0.00,0.00,0.00,0.00,-29.33,-30.80,-32.34,-33.95,-35.65,-37.43,-39.30,-41.27,-43.33,-45.50,-47.77,-50.16,-52.67,-55.30,-58.07,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
end_of_year,0.00,210.96,421.92,632.88,603.55,572.75,540.42,506.46,470.81,433.38,394.08,352.81,309.48,263.98,216.21,166.04,113.37,58.07,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
repayment_interest,0.00,0.00,0.00,0.00,-31.64,-30.18,-28.64,-27.02,-25.32,-23.54,-21.67,-19.70,-17.64,-15.47,-13.20,-10.81,-8.30,-5.67,-2.90,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
cash_flow_for_debt,0.00,0.00,0.00,0.00,156.16,158.31,160.48,162.69,-4.35,167.19,169.47,171.79,174.14,18.41,180.32,184.22,188.20,192.27,29.17,200.66,205.78,211.03,216.40,41.96,226.10,230.36,234.70,239.14,46.33,0.00,0.00
debt_service,0.00,0.00,0.00,0.00,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-60.97,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00
cash_after_debt_service,0.00,0.00,0.00,0.00,95.19,97.33,99.51,101.72,-65.32,106.21,108.50,110.82,113.16,-42.56,119.35,123.25,127.23,131.29,-31.81,200.66,205.78,211.03,216.40,41.96,226.10,230.36,234.70,239.14,46.33,0.00,0.00


Project reserves

In [16]:
from finances.project_reserves_and_equity import project_reserves

df_project_reserves = project_reserves(**onel_parameters)
df_project_reserves.style.format(precision=2)

project_reserves,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
contingency_injection,-126.58,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
contingency_reserve_balance,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,126.58,0.00
contingency_reserve_to_dividents,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,126.58


Equity funding

In [17]:
from finances.project_reserves_and_equity import equity_funding

df_equity_funding = equity_funding(df_decommissioning_phase,
               df_debt_and_loan_part2,
               df_construction_phase,
               **onel_parameters)

df_equity_funding.style.format(precision=2)

equity_funding,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
equity_injection,-126.58,-210.96,-210.96,-210.96,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-22.48,-22.93
dividents_results,0.00,0.00,0.00,0.00,95.19,97.33,99.51,101.72,-65.32,106.21,108.50,110.82,113.16,-42.56,119.35,123.25,127.23,131.29,-31.81,200.66,205.78,211.03,216.40,41.96,226.10,230.36,234.70,239.14,46.33,0.00,126.58
equity_cash_flow_result,-126.58,-210.96,-210.96,-210.96,95.19,97.33,99.51,101.72,-65.32,106.21,108.50,110.82,113.16,-42.56,119.35,123.25,127.23,131.29,-31.81,200.66,205.78,211.03,216.40,41.96,226.10,230.36,234.70,239.14,46.33,-22.48,103.65


Present value of cash flows

In [18]:
from finances.present_value_and_cumulative_cashflows import present_value_cashflows

df_present_value_cashflows = present_value_cashflows(df_construction_phase,
                                                     df_operational_phase,
                                                     df_decommissioning_phase,
                                                     df_equity_funding,
                                                     **onel_parameters)

df_present_value_cashflows.style.format(precision=2)

present_value,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
sum_net_project_cash_flows,0.00,-421.92,-421.92,-421.92,181.85,185.25,188.72,192.26,-4.35,199.53,203.26,207.07,210.95,18.41,220.83,226.91,233.15,239.54,29.17,252.83,259.73,266.80,274.05,41.96,287.11,292.85,298.71,304.68,46.33,-22.48,-22.93
present_value_net_cashflows,0.00,-385.31,-351.88,-321.35,126.49,117.68,109.48,101.86,-2.10,88.16,82.02,76.31,70.99,5.66,61.98,58.16,54.58,51.21,5.69,45.08,42.29,39.67,37.21,5.20,32.52,30.29,28.22,26.28,3.65,-1.62,-1.51
cumulative_value_net_cashflows,0.00,-385.31,-737.20,-1058.55,-932.06,-814.38,-704.90,-603.04,-605.15,-516.99,-434.97,-358.66,-287.67,-282.01,-220.03,-161.87,-107.29,-56.08,-50.39,-5.31,36.98,76.65,113.86,119.07,151.58,181.87,210.09,236.37,240.02,238.40,236.90
present_value_equity_cashflows,-126.58,-192.66,-175.94,-160.68,66.21,61.83,57.73,53.89,-31.60,46.93,43.78,40.84,38.08,-13.08,33.50,31.59,29.78,28.07,-6.21,35.78,33.51,31.38,29.39,5.20,25.61,23.83,22.17,20.63,3.65,-1.62,6.81


Cumulative equity and debt cash flows

In [19]:
from finances.present_value_and_cumulative_cashflows import cumulative_equity_and_debt_cashflows

df_cumulative_equity_and_debt_cashflows = cumulative_equity_and_debt_cashflows(df_equity_funding,
                                                                               df_debt_and_loan_part1,
                                                                               **onel_parameters)
df_cumulative_equity_and_debt_cashflows.style.format(precision=2)

cumulative equity and debt,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
cumulative_equity_cashflow,-126.58,-337.53,-548.49,-759.45,-664.26,-566.93,-467.42,-365.70,-431.02,-324.81,-216.31,-105.49,7.68,-34.89,84.46,207.71,334.93,466.23,434.42,635.08,840.86,1051.89,1268.29,1310.26,1536.35,1766.71,2001.42,2240.55,2286.89,2264.41,2368.06
cumulative_debt_cashflow,-0.00,-210.96,-421.92,-632.88,-603.55,-572.75,-540.42,-506.46,-470.81,-433.38,-394.08,-352.81,-309.48,-263.98,-216.21,-166.04,-113.37,-58.07,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00


Discounted cash flows for levelized cost

In [20]:
def discounted_cashflows_for_levelized_cost(df_construction_phase,
                                            df_operational_phase,
                                            df_decommissioning_phase,
                                            df_taxes_and_profits_part2,
                                            df_project_reserves,
                                            **kwargs):
    '''This function creates a dataframe that contains the discounted cashflows for levelized cost calculations'''

    # Using **kwargs we can pass the dictionary of parameters from the input data file into the function here

    # 1. Get the parameters we need
    wacc = kwargs['wacc']

    # 2. Get the timelines
    calendar_years = construct_calendar_year_list(**kwargs)
    business_case_years = construct_business_case_year_list(**kwargs)
    operational_years = construct_operations_years_list(**kwargs)

    # 3. Construct df
    row_names = [
        'capex', 'opex', 'purchase_electricity_ppa', 'purchase_electricity_grid', 
        'electricity_grid_connection', 'h2_storage_costs', 'entry_tariff_h2_network',
        'electricity_tax', 'stack_replacement', 
        'decommissioning', 'interest_costs', 'contingency', 'tax_expenses', 
        'h2_revenues', 'hwi_revenues', 'total_hydrogen_produced'
    ]
    df_discounted_levelized = pd.DataFrame(0.0, index=row_names, columns=calendar_years)
    df_discounted_levelized.columns.name = "discounted cashflows for levelized cost"
    
    # 4. Set up the discount term and calculate levelized costs and revenues
    discount_term = (1 + wacc) ** np.array(business_case_years)

# LEVELIZED COSTS

    df_discounted_levelized.loc['capex'] = df_construction_phase.loc['capex'] / discount_term
    df_discounted_levelized.loc['opex'] = df_operational_phase.loc['opex'] / discount_term
    df_discounted_levelized.loc['purchase_electricity_ppa'] = df_operational_phase.loc['purchase_electricity_ppa'] / discount_term
    df_discounted_levelized.loc['purchase_electricity_grid'] = df_operational_phase.loc['purchase_electricity_grid'] / discount_term
    df_discounted_levelized.loc['electricity_grid_connection'] = df_operational_phase.loc['electricity_grid_connection'] / discount_term
    df_discounted_levelized.loc['h2_storage_costs'] = df_operational_phase.loc['h2_storage_costs'] / discount_term
    df_discounted_levelized.loc['entry_tariff_h2_network'] = df_operational_phase.loc['entry_tariff_h2_network'] / discount_term
    df_discounted_levelized.loc['electricity_tax'] = df_operational_phase.loc['electricity_tax'] / discount_term
    df_discounted_levelized.loc['stack_replacement'] = df_operational_phase.loc['stack_replacement'] / discount_term
    df_discounted_levelized.loc['decommissioning'] = df_decommissioning_phase.loc['decommissioning_costs'] / discount_term
    df_discounted_levelized.loc['interest_costs'] = df_taxes_and_profits_part2.loc['interest_costs'] / discount_term

    # contingency (injection + divident return)
    df_discounted_levelized.loc['contingency'] = (
        (df_project_reserves.loc['contingency_injection'] 
         + df_project_reserves.loc['contingency_reserve_to_dividents']) / discount_term)

    # tax expenses
    df_discounted_levelized.loc['tax_expenses'] = df_taxes_and_profits_part2.loc['tax_expenses'] / discount_term


# LEVELIZED REVENUES

    df_discounted_levelized.loc['h2_revenues'] = df_operational_phase.loc['h2_revenues'] / discount_term
    df_discounted_levelized.loc['hwi_revenues'] = df_operational_phase.loc['hwi_revenues'] / discount_term


    # calculate discounted production   
    production = onel_h2_sold_to_hpa.loc['onel_h2_sold_to_hpa', calendar_years]
    df_discounted_levelized.loc['total_hydrogen_produced'] = np.where(
        production.index.isin(operational_years), 
        production / discount_term, 
        0.0
    )


    return df_discounted_levelized


In [21]:
df_discounted_cashflows_for_levelized_cost = discounted_cashflows_for_levelized_cost(df_construction_phase,
                                                                                     df_operational_phase,
                                                                                     df_decommissioning_phase,
                                                                                     df_taxes_and_profits_part2,
                                                                                     df_project_reserves,
                                                                                     **onel_parameters)
df_discounted_cashflows_for_levelized_cost.style.format(precision=2)

discounted cashflows for levelized cost,2027,2028,2029,2030,2031,2032,2033,2034,2035,2036,2037,2038,2039,2040,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050,2051,2052,2053,2054,2055,2056,2057
capex,0.00,-385.31,-351.88,-321.35,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
opex,0.00,0.00,0.00,0.00,-27.79,-25.36,-23.13,-21.09,-19.22,-17.50,-15.94,-14.50,-13.18,-11.98,-11.16,-10.40,-9.68,-9.02,-8.40,-7.83,-7.29,-6.79,-6.33,-5.89,-5.49,-5.11,-4.76,-4.44,-4.13,0.00,0.00
purchase_electricity_ppa,0.00,0.00,0.00,0.00,-71.94,-66.98,-62.36,-58.06,-54.06,-50.33,-46.86,-43.63,-40.62,-37.82,-32.91,-28.50,-24.53,-20.98,-17.80,-14.95,-12.42,-10.15,-8.14,-6.36,-5.93,-5.52,-5.14,-4.79,-4.46,0.00,0.00
purchase_electricity_grid,0.00,0.00,0.00,0.00,-7.09,-7.74,-8.26,-8.68,-9.00,-9.23,-9.39,-9.49,-9.53,-9.51,-8.02,-6.68,-5.49,-4.43,-3.49,-2.66,-1.92,-1.27,-0.71,-0.21,-0.20,-0.18,-0.17,-0.16,-0.15,0.00,0.00
electricity_grid_connection,0.00,0.00,0.00,0.00,-54.32,-50.60,-47.14,-43.91,-40.90,-38.10,-35.49,-33.06,-30.79,-28.69,-26.72,-24.89,-23.19,-21.60,-20.12,-18.74,-17.46,-16.26,-15.15,-14.11,-13.14,-12.24,-11.40,-10.62,-9.90,0.00,0.00
h2_storage_costs,0.00,0.00,0.00,0.00,-1.41,-1.31,-1.22,-1.14,-1.06,-0.99,-0.92,-0.86,-0.80,-0.75,-0.71,-0.68,-0.65,-0.62,-0.59,-0.56,-0.53,-0.51,-0.48,-0.46,-0.43,-0.40,-0.37,-0.34,-0.32,0.00,0.00
entry_tariff_h2_network,0.00,0.00,0.00,0.00,-3.98,-3.70,-3.45,-3.21,-2.99,-2.79,-2.60,-2.42,-2.25,-2.10,-1.96,-1.82,-1.70,-1.58,-1.47,-1.37,-1.28,-1.19,-1.11,-1.03,-0.96,-0.90,-0.83,-0.78,-0.72,0.00,0.00
electricity_tax,0.00,0.00,0.00,0.00,-4.32,-4.02,-3.74,-3.49,-3.25,-3.03,-2.82,-2.63,-2.45,-2.28,-2.12,-1.98,-1.84,-1.72,-1.60,-1.49,-1.39,-1.29,-1.20,-1.12,-1.04,-0.97,-0.91,-0.84,-0.79,0.00,0.00
stack_replacement,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-96.87,0.00,0.00,0.00,0.00,-60.39,0.00,0.00,0.00,0.00,-42.35,0.00,0.00,0.00,0.00,-29.70,0.00,0.00,0.00,0.00,-20.83,0.00,0.00
decommissioning,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,-1.62,-1.51


Levelized costs

In [22]:
def levelized_cost_and_revenues(df_discounted_cashflows_for_levelized_cost,
                                **kwargs):
    ''' This function creates a dataframe that contains the levelized costs and revenues'''
    
    # Using **kwargs we can pass the dictionary of parameters from the input data file into the function here

    # 1. Get the row names, except for total_hydrogen_produced
    row_names_levelized_cost = df_discounted_cashflows_for_levelized_cost.index.tolist()
    row_names_levelized_cost.remove('total_hydrogen_produced')

    # 2. Construct df
    df_levelized_cost = pd.DataFrame(0.0, index=row_names_levelized_cost, columns=['cost','revenues'])
    df_levelized_cost.columns.name = "levelized cost calculations"

    # 3. Calculate total discounted hydrogen production
    total_hydrogen_production_sum = df_discounted_cashflows_for_levelized_cost.loc['total_hydrogen_produced'].sum()

    # 4. Calculate the sum for every row
    row_sums = df_discounted_cashflows_for_levelized_cost.loc[row_names_levelized_cost].sum(axis=1)

    # 5. Arrange data over 'revenues' (if positive) and 'costs' (if negative)
    # use factor 1E6 because cost&revenues are in MEUR
    df_levelized_cost['revenues'] = np.where(row_sums>0, row_sums * 1E6 / total_hydrogen_production_sum, 0.0)
    df_levelized_cost['cost'] = np.where(row_sums<0, -row_sums * 1E6 / total_hydrogen_production_sum, 0.0)

    # 6. Calculate total costs and revenues
    total_costs = df_levelized_cost['cost'].sum()
    total_revenues = df_levelized_cost['revenues'].sum()

    # 7. Calculate profits or unprofitable gap
    df_levelized_cost.loc['profits'] = [0.0, 0.0]
    if total_revenues > total_costs:
        df_levelized_cost.loc['profits', 'cost'] = total_revenues - total_costs

    df_levelized_cost.loc['unprofitable_gap'] = [0.0, 0.0]
    if total_costs > total_revenues: 
        df_levelized_cost.loc['unprofitable_gap', 'revenues'] = total_costs - total_revenues


    return df_levelized_cost


In [23]:
df_levelized_cost_and_revenues = levelized_cost_and_revenues(df_discounted_cashflows_for_levelized_cost,
                                                             **onel_parameters)
df_levelized_cost_and_revenues.style.format(precision=2)

levelized cost calculations,cost,revenues
capex,80.73,0.00
opex,22.61,0.00
purchase_electricity_ppa,56.07,0.00
purchase_electricity_grid,9.43,0.00
electricity_grid_connection,50.22,0.00
h2_storage_costs,1.38,0.00
entry_tariff_h2_network,3.68,0.00
electricity_tax,3.99,0.00
stack_replacement,19.08,0.00
decommissioning,0.24,0.00


Project KPI's

In [24]:
from finances.kpis import project_kpi

df_project_kpi = project_kpi(df_present_value_cashflows,
                             df_taxes_and_profits_part2,
                             df_construction_phase,
                             **onel_parameters)

df_project_kpi.style.format(precision=2)

Project KPIs,Value,Unit
net_present_value,236.90,MEUR
internal_rate_of_return,11.76,%
return_on_investment,190.67,%
payback_period,13.11,years
discounted_return_on_investment,22.67,%
discounted_payback_period,20.38,years


Equity KPI's

In [25]:
from finances.kpis import equity_kpi

df_project_kpi = equity_kpi(df_equity_funding,
                            df_present_value_cashflows,
                             **onel_parameters)

df_project_kpi.style.format(precision=2)

Equity KPIs,Value,Unit
net_present_value,61.81,MEUR
internal_rate_of_return,10.33,%
return_of_investment,394.22,%
payback_period,6.34,years
discounted_return_on_investment,34.61,%
discounted_payback_period,18.57,years


Output KPI's

In [26]:
from finances.kpis import output_kpi

df_project_kpi = output_kpi(df_levelized_cost_and_revenues,
                            **onel_parameters)

df_project_kpi.style.format(precision=2)

Output KPIs,Value,Unit
levelized_cost,283.44,Eur/MWh
levelized_revenues,265.48,Eur/MWh
levelized_profits,-17.97,Eur/MWh
